In [82]:
import mlflow
import pandas
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.datasets import fetch_california_housing

In [83]:
mlflow.set_experiment("Random Forest")

<Experiment: artifact_location='/Users/benjaminbrooke/PycharmProjects/MLOps/Complete_MLOps_Bootcamp_With_10+_End_To_End_ML_Projects/3.MLFlow/mlruns/5', creation_time=1776524826073, experiment_id='5', last_update_time=1776524826073, lifecycle_stage='active', name='Random Forest', tags={}, trace_location=None, workspace='default'>

In [84]:
mlflow.set_tracking_uri("sqlite:////Users/benjaminbrooke/PycharmProjects/MLOps/Complete_MLOps_Bootcamp_With_10+_End_To_End_ML_Projects/3.MLFlow/mlflow.db")

#mlflow ui --backend-store-uri sqlite:////Users/benjaminbrooke/PycharmProjects/MLOps/Complete_MLOps_Bootcamp_With_10+_End_To_End_ML_Projects/3.MLFlow/mlflow.db

In [85]:
housing = fetch_california_housing()
X = housing.data
y = housing.target

print(X)
print(y)

[[   8.3252       41.            6.98412698 ...    2.55555556
    37.88       -122.23      ]
 [   8.3014       21.            6.23813708 ...    2.10984183
    37.86       -122.22      ]
 [   7.2574       52.            8.28813559 ...    2.80225989
    37.85       -122.24      ]
 ...
 [   1.7          17.            5.20554273 ...    2.3256351
    39.43       -121.22      ]
 [   1.8672       18.            5.32951289 ...    2.12320917
    39.43       -121.32      ]
 [   2.3886       16.            5.25471698 ...    2.61698113
    39.37       -121.24      ]]
[4.526 3.585 3.521 ... 0.923 0.847 0.894]


In [86]:
data = pd.DataFrame(housing.data,columns=housing.feature_names)
data.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [87]:
data["price"] = housing.target

In [88]:
data.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,price
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [89]:
from urllib.parse import urlparse

In [90]:
X = data.drop("price", axis=1)
y = data["price"]

In [91]:
X.head(5)

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [92]:
 X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

In [93]:
from mlflow.models import infer_signature

signature = infer_signature(X_train,y_train)

para_grid = {
    "n_estimators": [100,200],
    "max_depth": [4,5,None],
    "min_samples_split": [2,5],
    "min_samples_leaf": [1,2]}

In [94]:
def hyperparameter_tuning(X_train,y_train,para_grid):
    rf = RandomForestRegressor()
    grid_search = GridSearchCV(estimator=rf,
                              param_grid=para_grid,
                              scoring='neg_mean_squared_error',
                              n_jobs=-1,
                              verbose=2,
                              cv=3)
    grid_search.fit(X_train, y_train)
    return grid_search

In [96]:
with mlflow.start_run():

    grid_search = hyperparameter_tuning(X_train,y_train,para_grid)

    best_model = grid_search.best_estimator_

    y_pred =  best_model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)

    mlflow.log_param("best_n_estimators",grid_search.best_params_["n_estimators"])
    mlflow.log_param("best_max_depth",grid_search.best_params_["max_depth"])
    mlflow.log_param("best_min_samples_split",grid_search.best_params_["min_samples_split"])
    mlflow.log_param("best_min_samples_leaf",grid_search.best_params_["min_samples_leaf"])

    mlflow.log_metric("mse",mse)

    model_info=mlflow.sklearn.log_model(
        sk_model=best_model,
        name="Random Forest",
        signature=signature,
        registered_model_name="Random Forest")



Fitting 3 folds for each of 24 candidates, totalling 72 fits
[CV] END max_depth=4, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   2.5s
[CV] END max_depth=4, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   2.6s
[CV] END max_depth=4, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   2.7s
[CV] END max_depth=4, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   2.7s
[CV] END max_depth=4, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   2.6s
[CV] END max_depth=4, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   4.9s
[CV] END max_depth=4, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   2.6s
[CV] END max_depth=4, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   5.1s
[CV] END max_depth=4, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   5.2s
[CV] END max_depth=4, min_samples_leaf=

2026/04/18 17:21:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'Random Forest'.
Created version '1' of model 'Random Forest'.
